In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Step 1: Generate the Half-Moons Dataset
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)  # 300 samples

# Step 2: Split into Labeled (Small) and Unlabeled (Large) Data
X_labeled, X_unlabeled, y_labeled, _ = train_test_split(X, y, train_size=20, random_state=42)  # Only 20 labeled

# Step 3: Train Initial Supervised Model (SVM)
model = SVC(probability=True, kernel='rbf', C=1)  # SVM with RBF kernel
model.fit(X_labeled, y_labeled)

# Step 4: Self-Training Loop
max_iterations = 5  # Number of self-training steps
threshold = 0.9     # Confidence threshold for pseudo-labeling

for i in range(max_iterations):
    # Predict probabilities on unlabeled data
    probs = model.predict_proba(X_unlabeled)
    max_probs = np.max(probs, axis=1)  # Get max probability per sample
    pseudo_labels = np.argmax(probs, axis=1)  # Get predicted class

    # Select high-confidence predictions
    confident_idx = np.where(max_probs > threshold)[0]

    if len(confident_idx) == 0:  # Stop if no confident samples
        print(f"Iteration {i+1}: No confident samples found. Stopping self-training.")
        break

    # Add high-confidence predictions to the labeled dataset
    X_labeled = np.vstack([X_labeled, X_unlabeled[confident_idx]])
    y_labeled = np.hstack([y_labeled, pseudo_labels[confident_idx]])

    # Remove labeled samples from unlabeled set
    X_unlabeled = np.delete(X_unlabeled, confident_idx, axis=0)

    # Retrain the model with the updated labeled dataset
    model.fit(X_labeled, y_labeled)

    print(f"Iteration {i+1}: Added {len(confident_idx)} confident samples.")

# Step 5: Plot Function
def plot_data(X, y, labeled_idx, title):
    plt.scatter(X[:, 0], X[:, 1], c='gray', edgecolors='k', alpha=0.3, label="Unlabeled")
    plt.scatter(X[labeled_idx, 0], X[labeled_idx, 1], c=y[labeled_idx], cmap='coolwarm', edgecolors='k', s=100, label="Labeled")
    plt.title(title)
    plt.xlabel("X1")
    plt.ylabel("X2")
    plt.legend()
    plt.show()

# Plot Original Data (Before Self-Training)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plot_data(X, y, np.arange(20), "Original Data (Labeled + Unlabeled)")

# Plot Final Decision Boundary After Self-Training
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 2)
x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.2)
plt.scatter(X_labeled[:, 0], X_labeled[:, 1], c=y_labeled, cmap='coolwarm', edgecolors='k', s=100)
plt.title("Final Decision Boundary After Self-Training")
plt.xlabel("X1")
plt.ylabel("X2")
plt.show()
